In [ ]:

import torch

# Создание разных тензоров
a = torch.tensor([1.0, 2.0, 3.0])          # 1D-тензор из списка
b = torch.zeros((2, 3), dtype=torch.int64) # матрица 2×3 из нулей (64-битные целые)
c = torch.rand(3, 3)                        # случайный тензор 3×3, равномерное распределение


print(a, a.dtype)  # tensor([1., 2., 3.]) torch.float32

print(b, b.dtype)  # tensor([[0, 0, 0],
                   #         [0, 0, 0]]) torch.int64

print(c)           # тензор из случайных чисел размером 3×3



tensor([1., 2., 3.]) torch.float32
tensor([[0, 0, 0],
        [0, 0, 0]]) torch.int64
tensor([[0.6900, 0.1022, 0.4517],
        [0.0202, 0.0806, 0.8831],
        [0.8001, 0.4701, 0.5318]])


In [2]:
x = torch.tensor([1.0, -2.0, 3.0])
y = torch.exp(x)    # элемент-wise экспонента
z = x + y           # сложение поэлементно
print(z)

tensor([ 3.7183, -1.8647, 23.0855])


# Auto Gradient

In [3]:
x = torch.ones(2, 2, requires_grad=True)  # создаём тензор, указывая, что нужны градиенты
y = x * 2

z = y.mean()      # свёртка: усредняем все элементы
z.backward()      # выполняем обратный проход (backpropagation)
print(x.grad)     # смотрим градиент dz/dx

tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])


In [4]:
import torch

from torch import nn


model = nn.Linear(in_features=10, out_features=5)
print("Весовой тензор W:", model.weight.shape)   # torch.Size([5, 10])
print("Вектор смещений b:", model.bias.shape)    # torch.Size([5])


# Проверим, что по умолчанию требуется вычисление градиентов
print(model.weight.requires_grad)
print(model.bias.requires_grad) 

Весовой тензор W: torch.Size([5, 10])
Вектор смещений b: torch.Size([5])
True
True


In [5]:
"""ReLU"""

import torch

from torch import nn


relu = nn.ReLU()
x = torch.tensor([[-1.0, 0.0, 2.0], [3.0, -5.0, 1.0]])
y = relu(x)
print(y)

tensor([[0., 0., 2.],
        [3., 0., 1.]])


In [6]:
import torch

from torch import nn


# Предположим, что мы уже создали в __init__ следующие слои:
fc1 = nn.Linear(10, 5)     # из 10 входов в 5 нейронов
relu = nn.ReLU()
fc2 = nn.Linear(5, 1)      # из 5 входов в 1 нейрон


# А вот упрощённая функция forward, написанная вне класса:
def forward_pass(x):
    x1 = fc1(x)            # применили первый линейный слой
    x2 = relu(x1)          # применили ReLU
    x3 = fc2(x2)           # применили второй линейный слой
    return x3


# Создадим тестовый входной батч: batch_size=3, in_features=10

input_tensor = torch.randn(3, 10)
output_tensor = forward_pass(input_tensor)
print(input_tensor)
print(output_tensor)  # tensor of shape [3, 1]

tensor([[ 0.5328,  1.2266,  1.0831,  0.9159,  0.9612, -1.2117,  0.5277,  1.3101,
         -0.5900, -0.9981],
        [ 0.5307,  0.9294,  0.2980,  0.1904, -0.6434, -1.6982,  0.2418,  1.0779,
         -2.0313,  0.0571],
        [ 1.4869,  1.2205,  0.3215, -0.3082,  0.7656,  0.2191, -0.6546,  2.1442,
         -1.6075, -2.0022]])
tensor([[-0.1873],
        [-0.1855],
        [-0.4545]], grad_fn=<AddmmBackward0>)


# Что нужно сделать
* Создайте свой собственный класс нейросети. 


Внутри в __init__:
* self.fc1: линейный слой из 10 входов в 5 выходов,
* self.relu: функцию активации ReLU,
* self.fc2: линейный слой из 5 входов в 1 выход.

А в forward опишите логику:
* пропустить вход x через self.fc1,
* применить self.relu к результату,
* затем передать в self.fc2 и вернуть итоговый тензор.



In [10]:
from torch import nn

class SimpleNN(nn.Module):
    def __init__(self):
        # Базовая инициализация nn.Module
        super().__init__()
        # Линейный слой: принимает тензор размера [..., 10], выдаёт [..., 5]
        self.fc1 = nn.Linear(in_features=10, out_features=5)
        # Функция активации ReLU
        self.relu = nn.ReLU()
        # Линейный слой: принимает [..., 5], выдаёт [..., 1]
        self.fc2 = nn.Linear(in_features=5, out_features=1)


    def forward(self, x):
        # x - тензор размерности [batch_size, 10]
        x = self.fc1(x)          # теперь x → [batch_size, 5]
        # Примените ReLU
        x = self.relu(x)
        # Линейный слой x → [batch_size, 1]      
        x = self.fc2(x)
        return x                 # возвращаем тензор прогнозов


# Пример использования:
model = SimpleNN()
print(model)


# Можно сразу посмотреть, сколько параметров зарегистрировано:
total_params = sum(p.numel() for p in model.parameters())
print(f"Всего параметров: {total_params}")

SimpleNN(
  (fc1): Linear(in_features=10, out_features=5, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=5, out_features=1, bias=True)
)
Всего параметров: 61


# DataLoader, DataSet

In [11]:

from torch.utils.data import Dataset


class MyDataset(Dataset):
    def __init__(self, data, targets):
        self.data = data
        self.targets = targets


    def __len__(self):
        return len(self.data)


    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx]


# Создадим датасет из списков чисел
dataset = MyDataset(data=list(range(10)), targets=[2*x for x in range(10)])
print(len(dataset), dataset[0])  # 10, (0, 0)
print(dataset[1])            

10 (0, 0)
(1, 2)


In [12]:

from torch.utils.data import DataLoader


class MyDataset(Dataset):
    def __init__(self, data, targets):
        self.data = data
        self.targets = targets


    def __len__(self):
        return len(self.data)


    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx]


# Создадим датасет из списков чисел
dataset = MyDataset(data=list(range(10)), targets=[2*x for x in range(10)])


dataloader = DataLoader(dataset, batch_size=3, shuffle=True, num_workers=0)
for batch_data, batch_target in dataloader:
    print(batch_data, batch_target)
    break  # посмотрим только первый батч

tensor([5, 7, 2]) tensor([10, 14,  4])


In [ ]:
"""Реализация DataSet"""


# 1. Реализуйте класс ToyDataset

class ToyDataset(Dataset):
    def __init__(self, data, targets):
        # Сохраните data и targets в атрибуты
        self.data = data
        self.targets = targets


    def __len__(self):
        # Верните длину датасета
        return len(self.data)


    def __getitem__(self, idx):
        # Верните пару (вход, метка), соответствующую индексу idx
        return self.data[idx]


# 2. Подготовьте входные списки
raw_data = list(range(10)) # data = [0, 1, 2, ..., 9]
raw_targets = [2 * x for x in raw_data]  # targets = [0, 2, 4, ..., 18]


# 3. Создайте объект ToyDataset

dataset = ToyDataset(# Ваш код здесь)


# 4. Оберните датасет в DataLoader с batch_size=4 и shuffle=True

dataloader = DataLoader(# Ваш код здесь)


# Переберите первые два батча и выведите их на экран
for batch_idx, (batch_inputs, batch_targets) in enumerate(dataloader):
    print(f"Батч {batch_idx + 1}:")
    print("Inputs: ", batch_inputs)
    print("Targets:", batch_targets)
    if batch_idx == 1:  # остановимся после второго батча
        break